## Question 6 - *Do tasks with a low scheduling class have a higher probability of being evicted?*

Similar to the previous question, we need to check the distribution of *evicted_tasks* according to their *scheduling_class*. <br>
In the *task_events* dataset, the identifier of a taks is the couple (job_id, task_idx). By filtering the dataset to only keep the evicted_tasks and then only keeping the scheduling_class, we can make the distribution of the evictions according to the scheduling_class. <br>
As it appears, low scheduling classes tend to have more evictions than tasks with higher scheduling classes.

In [ ]:
# Initialising Spark 
from pyspark import SparkContext
sc = SparkContext("local[*]")

task_events_1 = sc.textFile("./data/task_events/part-00060-of-00500.csv.gz")
task_events_2 = sc.textFile("./data/task_events/part-00061-of-00500.csv.gz")
task_events_3 = sc.textFile("./data/task_events/part-00062-of-00500.csv.gz")
task_events_4 = sc.textFile("./data/task_events/part-00063-of-00500.csv.gz")
task_events_4 = sc.textFile("./data/task_events/part-00064-of-00500.csv.gz")

task_events= task_events_1.union(task_events_2).union(task_events_3).union(task_events_4)

evicted_tasks = (
    task_events_1.union(task_events_2).union(task_events_3).union(task_events_4)
    .map(lambda line: line.split(","))
    .filter(lambda x: x[5] == '2')
    .map(lambda x: ((x[2],x[3]),(x[5],x[7])))  # (job_id, task_idx) (event_type,scheduling_class)
)
print(f"Number of evictions: {evicted_tasks.count()}")
distinct_evicted_tasks = evicted_tasks.distinct()
print(f"Number of distinct tasks evicted at least once: {distinct_evicted_tasks.count()}")

distribution_sclass_eviction = (
    distinct_evicted_tasks
    .map(lambda x: x[1][1])
    .map(lambda x: (x,1))
    .reduceByKey(lambda a, b: a + b)
)
print("\n" + "="*80)
print(f"Distribution of evited_tasks according to the scheduling_class: {distribution_sclass_eviction.take(10)}")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/01/14 23:14:33 WARN Utils: Your hostname, im2ag-mandelbrot, resolves to a loopback address: 127.0.1.1; using 152.77.81.20 instead (on interface ens18)
26/01/14 23:14:33 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/01/14 23:14:34 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Number of evictions: 30341


Number of distinct tasks evicted at least once: 19704

Distribution of evited_tasks according to the scheduling_class: [('2', 3714), ('1', 3566), ('0', 12230), ('3', 194)]
